# Study 935 — Value Averaging — the teardown

Rolling 36-month accumulation programmes on **SPY vs BIL**, both arms handed identical committed capital (a pre-funded buffer plus the same monthly contributions), idle money earning BIL's actual total return, one execution lag (sized at the decision month-end close, filled at the next day's close), 1 bp one-way on traded notional, no shorting and therefore no borrow. The headline quantity is terminal **whole-account** wealth in cents per dollar contributed.

Contents: the wealth gap and its HAC *t*, the two IRR measures, the cap-binding statistics, the era cut, four sweeps (value-path growth, buffer, cost, horizon), the exposure-matched control, the calibrated random-walk placebo, the sleeve cross-checks and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `9cce1b76d021`).

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'fp': '9cce1b76d021', 'n_windows': 193, 'first_start': '2007-05-31', 'last_val': '2026-06-01', 'horizon': 36, 'buffer': 6, 'growth': 0.0, 'cost_bps': 1.0, 'gap': -1.372, 'gap_med': -1.053, 'gap_sd': 2.113, 't_hac': -3.87, 't_nov': -0.98, 'n_nov': 6, 'boot_lo': -2.057, 'boot_hi': -0.742, 'win': 29.0, 'win_lo': 23.1, 'win_hi': 35.8, 'exc_va': 19.04, 'exc_dca': 20.41, 'irr_prog_va': 10.07, 'irr_prog_dca': 10.64, 'irr_eq_va': 14.9, 'irr_eq_dca': 13.97, 'irr_eq_edge': 0.92, 'inv_va': 0.634, 'inv_dca': 0.699, 'disp': 0.81, 'bind_windows': 6, 'bind_rate': 3.1, 'bind_month': 0.2, 'bind_months_total': 14, 'worst_month_shortfall': 3.32, 'worst_prog_shortfall': 6.57, 'va_notional': 34.66, 'dca_notional': 36.0, 'worst_gap': -7.38, 'worst_start': '2018-12-31', 'best_gap': 2.87, 'best_start': '2017-04-28', 'era_e_n': 104, 'era_e_gap': -1.518, 'era_e_t': -2.98, 'era_e_win': 23.1, 'era_l_n': 89, 'era_l_gap': -1.201, 'era_l_t': -2.5, 'era_l_win': 36.0, 'g0_gap': -1.372, 'g0_t': -3.87, 'g0_inv': 0.634, 'g4_gap': -0.644, 'g4_t': -2.02, 'g4_inv': 0.654, 'g8_gap': 0.096, 'g8_t': 0.33, 'g8_inv': 0.673, 'g12_gap': 0.834, 'g12_t': 3.23, 'g12_inv': 0.693, 'g12_bind': 13.0, 'buf0_gap': -1.675, 'buf0_t': -5.16, 'buf0_bind': 86.5, 'buf3_gap': -1.421, 'buf3_bind': 6.2, 'buf12_gap': -1.346, 'buf12_bind': 0.0, 'buf24_gap': -1.346, 'buf24_bind': 0.0, 'cost0_gap': -1.372, 'cost5_gap': -1.37, 'cost25_gap': -1.363, 'cost25_t': -3.91, 'h24_gap': -0.481, 'h24_t': -2.09, 'h24_win': 41.5, 'h60_gap': -5.154, 'h60_t': -7.1, 'h60_win': 5.9, 'h120_gap': -29.319, 'h120_t': -28.65, 'h120_win': 0.0, 'em_lambda': 0.8984, 'em_gap': 0.702, 'em_t': 3.05, 'em_win': 72.5, 'em_lo': 0.269, 'em_hi': 1.078, 'pl_n': 12, 'pl_mean': -0.249, 'pl_sd': 1.441, 'pl_lo': -2.838, 'pl_hi': 1.805, 'pl_z': 0.66, 'pl_above': 2, 'pl_p': 0.231, 'ief_gap': -0.165, 'ief_t': -1.7, 'ief_win': 48.7, 'qqq_gap': -3.469, 'qqq_t': -5.04, 'qqq_win': 17.1, 'syn_gap': 6.759, 'syn_t': 5.0, 'syn_win': 86.2, 'syn_em_gap': 8.185, 'syn_em_t': 4.26, 'syn_det_gap': -0.114}

## The headline

193 overlapping windows, starts 2007-05-31 → 2023-05-31. The HAC *t* is lag-truncated at 36 (the overlap span); the block bootstrap uses 36-window blocks; the non-overlapping check keeps every 36th window and therefore has only 6 observations, which is reported for honesty rather than for power.

In [2]:
print(f"gap (VA - DCA)   : {R['gap']:+.3f}c/dollar   median {R['gap_med']:+.3f}   sd {R['gap_sd']:.3f}")
print(f"HAC t (lag 36)   : {R['t_hac']:+.2f}   non-overlapping t {R['t_nov']:+.2f} (n={R['n_nov']})")
print(f"bootstrap 95% CI : [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]  -- entirely below zero")
print(f"VA win rate      : {R['win']:.1f}%  Wilson [{R['win_lo']:.1f}%, {R['win_hi']:.1f}%]")
print(f"excess-of-cash   : VA {R['exc_va']:+.2f}c vs DCA {R['exc_dca']:+.2f}c per dollar contributed")
print(f"mean equity wt   : VA {R['inv_va']:.3f} vs DCA {R['inv_dca']:.3f}   dispersion ratio {R['disp']:.3f}")
print(f"worst window {R['worst_gap']:+.2f}c ({R['worst_start']})   best {R['best_gap']:+.2f}c ({R['best_start']})")

gap (VA - DCA)   : -1.372c/dollar   median -1.053   sd 2.113
HAC t (lag 36)   : -3.87   non-overlapping t -0.98 (n=6)
bootstrap 95% CI : [-2.057, -0.742]  -- entirely below zero
VA win rate      : 29.0%  Wilson [23.1%, 35.8%]
excess-of-cash   : VA +19.04c vs DCA +20.41c per dollar contributed
mean equity wt   : VA 0.634 vs DCA 0.699   dispersion ratio 0.810
worst window -7.38c (2018-12-31)   best +2.87c (2017-04-28)


## The metric that makes VA look good

Two IRRs on the same programme. `equity_irr` discounts only the flows that touch the sleeve; `programme_irr` discounts the buffer and the contributions and values the whole account at the end. They disagree in sign.

> 💡 **In plain words:** the famous number measures the return on the money that happened to be invested, not the return on the money you had to commit.

In [3]:
print(f"equity-only IRR (Edleson)  : VA {R['irr_eq_va']:.2f}%/yr vs DCA {R['irr_eq_dca']:.2f}%/yr   ({R['irr_eq_edge']:+.2f} pp to VA)")
print(f"whole-programme IRR        : VA {R['irr_prog_va']:.2f}%/yr vs DCA {R['irr_prog_dca']:.2f}%/yr   ({R['irr_prog_va']-R['irr_prog_dca']:+.2f} pp to VA)")
for b, eq_va, pr_va, pr_dca in [(0, 14.73, 12.97, 13.92), (3, 14.87, 11.35, 12.03),
                                (6, 14.90, 10.07, 10.64), (12, 14.91, 8.27, 8.73),
                                (24, 14.91, 6.22, 6.56)]:
    print(f"  buffer {b:2d}xC : equity-only VA {eq_va:.2f}% (DCA {R['irr_eq_dca']:.2f}% always)   "
          f"whole-programme VA {pr_va:5.2f}% vs DCA {pr_dca:5.2f}%")
print('the left column barely moves while the programme underneath it changes completely')

equity-only IRR (Edleson)  : VA 14.90%/yr vs DCA 13.97%/yr   (+0.92 pp to VA)
whole-programme IRR        : VA 10.07%/yr vs DCA 10.64%/yr   (-0.57 pp to VA)
  buffer  0xC : equity-only VA 14.73% (DCA 13.97% always)   whole-programme VA 12.97% vs DCA 13.92%
  buffer  3xC : equity-only VA 14.87% (DCA 13.97% always)   whole-programme VA 11.35% vs DCA 12.03%
  buffer  6xC : equity-only VA 14.90% (DCA 13.97% always)   whole-programme VA 10.07% vs DCA 10.64%
  buffer 12xC : equity-only VA 14.91% (DCA 13.97% always)   whole-programme VA  8.27% vs DCA  8.73%
  buffer 24xC : equity-only VA 14.91% (DCA 13.97% always)   whole-programme VA  6.22% vs DCA  6.56%
the left column barely moves while the programme underneath it changes completely


## The mechanism — it is an exposure dial

The value path's growth rate is an **ASSUMPTION**, nowhere on the tape. Sweeping it moves VA's average equity weight, and the wealth gap follows it monotonically; the sign flips at roughly the point where the weights match. That is not a strategy parameter, it is a beta parameter.

In [4]:
for g, gap, t, inv in [(0, R['g0_gap'], R['g0_t'], R['g0_inv']),
                       (4, R['g4_gap'], R['g4_t'], R['g4_inv']),
                       (8, R['g8_gap'], R['g8_t'], R['g8_inv']),
                       (12, R['g12_gap'], R['g12_t'], R['g12_inv'])]:
    print(f"path growth {g:2d}%/yr : VA equity {inv:.3f} (DCA {R['inv_dca']:.3f})  gap {gap:+.3f}c  t {t:+.2f}")

path growth  0%/yr : VA equity 0.634 (DCA 0.699)  gap -1.372c  t -3.87
path growth  4%/yr : VA equity 0.654 (DCA 0.699)  gap -0.644c  t -2.02
path growth  8%/yr : VA equity 0.673 (DCA 0.699)  gap +0.096c  t +0.33
path growth 12%/yr : VA equity 0.693 (DCA 0.699)  gap +0.834c  t +3.23


## The funding constraint — how often the cap binds

The buffer is finite and the purchase is capped at what it can fund; the shortfall is recorded rather than silently borrowed. The binding is rare on average and brutally concentrated: all six binding programmes start between 2007-05 and 2007-10.

In [5]:
print(f"buffer {R['buffer']}xC : binds in {R['bind_windows']}/{R['n_windows']} windows ({R['bind_rate']:.1f}%), {R['bind_months_total']} binding months = {R['bind_month']:.2f}% of all rebalance months")
print(f"worst SINGLE-MONTH unfunded call : {R['worst_month_shortfall']:.2f} x the monthly contribution")
print(f"worst PROGRAMME-TOTAL shortfall  : {R['worst_prog_shortfall']:.2f} x (summed over that programme's binding months)")
for b, gap, bind in [(0, R['buf0_gap'], R['buf0_bind']), (3, R['buf3_gap'], R['buf3_bind']),
                     (6, R['gap'], R['bind_rate']), (12, R['buf12_gap'], R['buf12_bind']),
                     (24, R['buf24_gap'], R['buf24_bind'])]:
    print(f"  buffer {b:2d}xC : gap {gap:+.3f}c   cap binds in {bind:.1f}% of windows")
print('no buffer setting rescues the sign; a bigger buffer only dilutes both arms equally')

buffer 6xC : binds in 6/193 windows (3.1%), 14 binding months = 0.20% of all rebalance months
worst SINGLE-MONTH unfunded call : 3.32 x the monthly contribution
worst PROGRAMME-TOTAL shortfall  : 6.57 x (summed over that programme's binding months)
  buffer  0xC : gap -1.675c   cap binds in 86.5% of windows
  buffer  3xC : gap -1.421c   cap binds in 6.2% of windows
  buffer  6xC : gap -1.372c   cap binds in 3.1% of windows
  buffer 12xC : gap -1.346c   cap binds in 0.0% of windows
  buffer 24xC : gap -1.346c   cap binds in 0.0% of windows
no buffer setting rescues the sign; a bigger buffer only dilutes both arms equally


## Robustness — eras, costs, horizons, sleeves

Costs are a non-issue: VA trades *less* notional than DCA (34.7 vs 36.0 per 36 dollars contributed), because its sells net against its buys, so the gap barely moves out to 25 bp. The horizon sweep is the clearest reading of the mechanism — the longer the programme, the more the compounding equity-weight shortfall dominates.

In [6]:
print(f"era, start <2016 (n={R['era_e_n']}): gap {R['era_e_gap']:+.3f}c  t {R['era_e_t']:+.2f}  win {R['era_e_win']:.1f}%")
print(f"era, start >=2016 (n={R['era_l_n']}): gap {R['era_l_gap']:+.3f}c  t {R['era_l_t']:+.2f}  win {R['era_l_win']:.1f}%")
print(f"cost 0/5/25 bp : {R['cost0_gap']:+.3f} / {R['cost5_gap']:+.3f} / {R['cost25_gap']:+.3f}c (t {R['cost25_t']:+.2f} at 25bp)")
for h, gap, t, w in [(24, R['h24_gap'], R['h24_t'], R['h24_win']), (36, R['gap'], R['t_hac'], R['win']),
                     (60, R['h60_gap'], R['h60_t'], R['h60_win']), (120, R['h120_gap'], R['h120_t'], R['h120_win'])]:
    print(f"  horizon {h:3d}m : gap {gap:+8.3f}c  t {t:+7.2f}  VA wins {w:5.1f}%"
          + ('   <- t is DIRECTION only: HAC lag = horizon, and only 3 (60m) / 1 (120m) independent programmes exist' if h >= 60 else ''))
print(f"IEF sleeve : gap {R['ief_gap']:+.3f}c (t {R['ief_t']:+.2f})   QQQ sleeve : gap {R['qqq_gap']:+.3f}c (t {R['qqq_t']:+.2f})")
print('the gap scales with the sleeve return -- lowest on bonds, worst on the highest-drift sleeve')

era, start <2016 (n=104): gap -1.518c  t -2.98  win 23.1%
era, start >=2016 (n=89): gap -1.201c  t -2.50  win 36.0%
cost 0/5/25 bp : -1.372 / -1.370 / -1.363c (t -3.91 at 25bp)
  horizon  24m : gap   -0.481c  t   -2.09  VA wins  41.5%
  horizon  36m : gap   -1.372c  t   -3.87  VA wins  29.0%
  horizon  60m : gap   -5.154c  t   -7.10  VA wins   5.9%   <- t is DIRECTION only: HAC lag = horizon, and only 3 (60m) / 1 (120m) independent programmes exist
  horizon 120m : gap  -29.319c  t  -28.65  VA wins   0.0%   <- t is DIRECTION only: HAC lag = horizon, and only 3 (60m) / 1 (120m) independent programmes exist
IEF sleeve : gap -0.165c (t -1.70)   QQQ sleeve : gap -3.469c (t -5.04)
the gap scales with the sleeve return -- lowest on bonds, worst on the highest-drift sleeve


## The decisive cut — exposure-matched, then placebo'd

Dial DCA's monthly purchase down by lambda until the two arms carry the same mean equity weight; whatever survives is the contrarian *timing*. It is positive. **lambda is fitted in-sample on this very tape**, so that figure is an in-sample residual, not an out-of-sample result. Then run the identical exercise — same in-sample bisection — on twelve random walks calibrated to SPY's own drift, volatility and cash rate, i.e. worlds with **zero** predictability, and the same residual appears, because buying the dips of a random walk earns a mechanical rebalancing bonus. The real tape's residual sits inside that placebo spread. (The placebo tapes are homoskedastic, so if anything they *understate* the bonus a vol-clustered tape would hand a contrarian schedule — which cuts against, not for, the reading below.)

In [7]:
print(f"exposure-matched (lambda={R['em_lambda']:.4f}, fitted IN-SAMPLE): gap {R['em_gap']:+.3f}c  t {R['em_t']:+.2f}  "
      f"win {R['em_win']:.1f}%  CI [{R['em_lo']:+.3f}, {R['em_hi']:+.3f}]")
print(f"calibrated random-walk placebo ({R['pl_n']} paths): mean {R['pl_mean']:+.3f}c  sd {R['pl_sd']:.3f}  "
      f"range [{R['pl_lo']:+.3f}, {R['pl_hi']:+.3f}]")
print(f"-> the real tape's residual sits at z = {R['pl_z']:+.2f} of the no-predictability spread")
print(f"-> distribution-free: {R['pl_above']}/{R['pl_n']} zero-predictability paths beat it (one-sided p = {R['pl_p']:.3f})")

exposure-matched (lambda=0.8984, fitted IN-SAMPLE): gap +0.702c  t +3.05  win 72.5%  CI [+0.269, +1.078]
calibrated random-walk placebo (12 paths): mean -0.249c  sd 1.441  range [-2.838, +1.805]
-> the real tape's residual sits at z = +0.66 of the no-predictability spread
-> distribution-free: 2/12 zero-predictability paths beat it (one-sided p = 0.231)


## Live synthetic control — the machinery is unbiased

Planted transitory mean reversion (a sub-one variance ratio by construction): the exposure-matched race MUST find it. Zero-volatility tape: it must find nothing. **Synthetic data, not the real tape** — this proves the harness works, it never supports the stamp.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from value_avg import data, strategy as st
planted, truth = data.synthetic_daily(n_years=12, signal_strength=1.0, seed=935)
quiet,   _     = data.synthetic_daily(n_years=12, signal_strength=0.0, seed=935, vol_ann=0.0)
pl = st.exposure_matched_race(planted['asset'], planted['cash'], 36, tol=0.01,
                              max_iter=6, buffer_mult=6.0, cost_bps=1.0)
qt = st.exposure_matched_race(quiet['asset'], quiet['cash'], 36, tol=0.005,
                              max_iter=8, buffer_mult=6.0, cost_bps=1.0)
print('SYNTHETIC planted wobble (swing sd %.2f, half-life %.0fd):' % (truth['swing_eff'], truth['half_life_days']))
print('   exposure-matched gap %+.3fc  t %+.2f  VA wins %.1f%%' % (pl['gap_mean_cents'], pl['t_hac'], pl['va_win_rate']*100))
print('SYNTHETIC zero-volatility null:')
print('   exposure-matched gap %+.3fc  (must be ~0)' % qt['gap_mean_cents'])

SYNTHETIC planted wobble (swing sd 0.28, half-life 150d):
   exposure-matched gap +8.185c  t +4.26  VA wins 91.3%
SYNTHETIC zero-volatility null:
   exposure-matched gap -0.114c  (must be ~0)


## Verdict

- **Signal — None.** On identical committed capital the value-averaging programme ends **-1.372 cents per dollar contributed** behind plain DCA (HAC *t* = -3.87, bootstrap CI [-2.057, -0.742] entirely below zero, ahead in 29.0% of 193 programmes, negative in both eras, on IEF (-0.165) and on QQQ (-3.469), and at every horizon from 24 to 120 months). The sign is set by a **non-tape assumption** — the value path's growth rate, which is simply an equity-weight dial (0.634 at 0%/yr to 0.693 at 12%/yr against DCA's 0.699). Exposure-matched on an **in-sample** lambda, the residual is **+0.702c** — inside the spread a calibrated random walk produces with no predictability at all (z = +0.66; 2 of 12 such worlds beat it outright, one-sided *p* = 0.23), i.e. the rebalancing bonus, not timing. The synthetic control recovers a planted wobble (+8.19c, *t* = +4.26) and is silent on a zero-vol tape (-0.114c), so the harness is not the problem.
- **Tradability — Mirage.** The advertised edge survives only in the equity-only IRR (+0.92 pp/yr), a figure near-invariant to the buffer size (14.73% to 14.91% across the 0-24x sweep) while the whole-programme IRR halves over the same range — so the famous number cannot be measuring the programme; at the 6x default it is 10.07% vs 10.64%. Friction is not the obstacle (VA trades less notional than DCA and the gap is unchanged at 25 bp); the obstacles are the equity-weight give-up and a cash call that reached 3.3 monthly contributions in a single month (6.6 across that programme), in 2008, when it was least payable.